# Implementing tokenizer from scratch

## What is a tokenizer?
<div class="alert alert-block alert-success">
A component that breaks down raw text into smaller units called tokens, which are the basic building blocks that language models actually process.
</div>

For demonstration purpose, we work at small datasets. However, in the real world, it is common to preprocess a large amount of articles or books in order to train LLMs.

Firstly, let's read and inspect the text included here!

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


We are going to use Python regular expresseion library **re** for this implementation to help understand how tokenizer works.

Without further ado, let's go!

## Step 1: Creating tokens

In [2]:
import re
# Importing library

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


<div class="alert alert-block alert-success">
Let's use AI to break down everything happened here!
</div>
---

### 1. `import re`
- This imports Python’s **regular expression (regex)** library.
- Regex lets you define complex search/split patterns for strings.

---

### 2. `re.split(r'([.,:;?_"()\'`|--|\s)', raw_text)`
- **`re.split(pattern, string)`**: Splits `string` wherever the regex `pattern` matches.
- The pattern here is:
  ```
  r'([.,:;?_"()\'`|--|\s])'
  ```
  - `r''`: Raw string literal, so backslashes are treated literally.
  - `[ ... ]`: A **character class**, meaning “match any one of these characters.”
  - Inside the brackets:
    - `.,:;?_"()\'\`` → punctuation marks
    - `--` → includes double dashes
    - `\s` → matches any whitespace (spaces, tabs, newlines).
  - The **parentheses `( ... )`** around the character class make it a **capturing group**.  
    → This means the delimiters (punctuation/whitespace) are also returned in the split list, not discarded.

---

### 3. List comprehension
```python
[item.strip() for item in preprocessed if item.strip()]
```
- Iterates through each `item` in the list `preprocessed`.
- `item.strip()` removes leading/trailing whitespace.
- `if item.strip()` ensures only non-empty strings are kept.
- Result: a cleaned list of tokens (words and punctuation).

---

### 4. `print(preprocessed[:30])`
- `[:30]` is **list slicing**: takes the first 30 elements.
- Prints them for inspection.

---

### ⚡ Key Takeaways
- **Regex split with capturing group** → keeps punctuation as separate tokens.
- **List comprehension with `strip()`** → cleans and filters out empty strings.
- **Slice `[:30]`** → limits output for readability.

---
### Fun fact
The actual training of the large language model has a vocabulary list that contains sub-words, like:
`(tokenizer's amazing)` may consist of tokens like: `(["token", "izer", "'s", "amaz", "ing")`
For those new to code:
<div class="alert alert-block alert-warning">
Don't worry if the code is hard to understand. Everyone starts like a noob. This notebook is designed to fully understand LLM, feel free to ask any questions to AI if you have any doubts. What really matter is to understand the nuts and bolts of everything to eventually become confident.
</div>

## Step 2: Creating Token IDs
## What is Token ID?
<div class='alert alert-block alert-success'>
Token IDs are just basically an identification of tokens: the numeric index (integer) assigned to a particular token in the tokenizer's fixed vocabulary. It is the actual value that a language model receives and processes.
</div>

Let's see the vocablary size first!

In [3]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


Emmm, quite a few vocabulary in this dataset!

Now we will code a **vocabulary list**. It is a Python dictionary. Note that it has a result of `{token: integer}` as it is meant for **encoding** which will be covered later.

Since it is quite large, we are only going to print the first 25 entries in the list.

In [4]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [5]:
for i, item in enumerate(vocab.items()):#Loops through the dictionary entries (key, value pairs).
                                        #enumerate adds a counter i to track how many items have been seen.
    print(item)
    if i >= 24:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)


As we can see, the dictionary contains every unique integer labels that correspond to every individual tokens.

In LLM, the model will:
- Firstly **endode** the tokens into token IDs:
    
    splitting text into token, carrying out the string-to-integer mapping to produce token IDs into the dictionary which are going to be our vocabulary list.
- Then **decode** the tokens from their token IDs:
    
    Carrying out the integer-to-string mapping to produce token IDs back to text.

We are going to create a class for this, where we already have an intuition of what the class should include: **Encoder** and **Decoder**.
## 3. Coding a tokenizer class in Python

In [6]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        #Store the vocabulary as a class attribute for access in the encode and decode methods
        self.str_to_int = vocab # ENCODING
        #Note that vocab = {token:integer for integer, token in enumerate(all_words)}
        #Create an INVERSE vocabulary that maps token IDs back to the original text tokens
        self.int_to_str = {i:s for s, i in vocab.items()} # DECODING
    
    #Process input text into token IDs
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        #Splits input text into tokens using regex (punctuation + whitespace as separators)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        #Cleans tokens with strip() and filters out empty strings.
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    #Convert token IDs back into tokens
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        #Joins them with spaces.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        #Uses re.sub to remove unwanted spaces before punctuation (so you don’t get "Hello !", but "Hello!").
        return text

Create an instance and try out!

In [7]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
#Notice this text snippet is directly copy pasted from the story
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


<div class='alert alert-block alert-success'>
Looks cool!

Try converting back to tokens:
</div>

In [8]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

<div class='alert alert-block alert-success'>
Not exactly the same but we can see that the decode method successfully converted the token IDs back into the original text.
</div>

But what about text that is **out of** the training dataset?

In [9]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

Nah. "Hello" is not in the vocabulary list as it is absent in the whole text, that's why it throws an `KeyError`.

<div class='alert alert-block alert-info'>
This highlights the need to consider large and diverse dataset to extend the vocabulary when working on LLMs.
</div>

## 4. Improving the tokenizer class
As we can see the current tokenizer is unable to encode words that is absent in the vocabulary.

Let's modify the tokenizer to handle unknown words! But, how?

You might be guessed it! We were talking about tokens all day long, so why not add special tokens?

We can modify the tokenizer to use an `<|unk|>` token if it encounters a word that is not part of the vocabulary. 

Btw we are also adding a token `<|endoftext|>` token between unrelated texts. In fact, when training GPT-like LLMs on multiple independent documents or books, it is common to insert a token before each document or book that follows a previous text source.

Let's start with extending the token list!

In [10]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

len(vocab.items())
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


Hmmm, exactly 2 more tokens than before. Good.
And the appended tokens are right there!

Let's enhance the code that we have already written before and try out!

In [11]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)                       
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        # Replace unknown words by <|unk|> tokens
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [12]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace"

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace


In [13]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131]

In [14]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>'

### Fun fact
The tokenizer of GPT model only uses an `<|endoftext|>` token for simplicity, it also doesn't use an `<|unk|>` token for out-of-vocabulary words. 

Instead, GPT models use a **byte pair encoding** tokenizer, which breaks down words into subword units, which we will cover later!
<div class='alert alert-block alert-success'>
As we can see, those <|unk|> tokens are those words that are not appeared in the text, while the <|endoftext|> token is the seperator between two seperated sentences.
</div>

## 5. Byte Pair Encoding (BPE)

Let me first tell you a truth: the BPE tokenizer was used to train LLMs such as GPT-2, 3, 4, and 5 that we use everyday☠️

But what on earth is BPE?
<div class ='alert alert-block alert-info'>
BPE is a sub-word tokenization algorithm that starts with individual bytes/characters and repeatedly merges the most frequently occuring adjacent pairs of tokens into new single tokens -- creating a vocabulary that contains both whole words and common subword fragments.
</div>

But this time, we are not going to implement from scratch! Instead we are going to use an existing Python open-source library called `tiktoken` (https://github.com/openai/tiktoken). 

This library implements the BPE algorithm very efficiently based on source code in Rust.

In [15]:
#Let's install it!
! pip3 install tiktoken

Defaulting to user installation because normal site-packages is not writeable


In [16]:
import importlib
import tiktoken
#Let's examine its version!
print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


Oh btw as you can tell from the install message, I am **Richard** who is writing this now! Name reveal already☠️☠️

Once installed, we can instantiate and try out the BPE tokenizer from `tiktoken` as follows:

In [17]:
tokenizer = tiktoken.get_encoding("gpt2")
# It's that easy

In [18]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

strings = tokenizer.decode(integers)

print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


We can make two noteworthy observations based on the token IDs and decoded text
above. 

<div class='alert alert-block alert-info'>
First, the <|endoftext|> token is assigned a relatively large token ID, namely,
50256. 

In fact, the BPE tokenizer, which was used to train models such as GPT-2, GPT-3,
and the original model used in ChatGPT, has a total vocabulary size of 50,257, with
<|endoftext|> being assigned the largest token ID.
</div>

<div class='alert alert-block alert-info'>
Second, the BPE tokenizer above encodes and decodes unknown words, such as
"someunknownPlace" correctly. 

The BPE tokenizer can handle any unknown word. How does
it achieve this without using <|unk|> tokens?
</div>

**Let us take another simple example to illustrate how the BPE tokenizer deals with unknown tokens**

In [19]:
integers = tokenizer.encode("Akwirw ier")
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


<div class="alert alert-block alert-warning">

The algorithm underlying BPE breaks down words that aren't in its predefined vocabulary
into smaller subword units or even individual characters.

The enables it to handle out-of-vocabulary words. 

So, thanks to the BPE algorithm, if the tokenizer encounters an
unfamiliar word during tokenization, it can represent it as a sequence of subword tokens or
characters.
</div>

Lastly, let's get all encodings of GPTs!

In [21]:
import tiktoken

# Initialize the encodings for GPT-2, GPT-3, and GPT-4
encodings = {
    "gpt2": tiktoken.get_encoding("gpt2"),
    "gpt3": tiktoken.get_encoding("p50k_base"),  # Commonly associated with GPT-3 models
    "gpt4": tiktoken.get_encoding("cl100k_base"),  # Used for GPT-3.5 and GPT-4
    "gpt4o or later": tiktoken.get_encoding("o200k_base"), # Used for all modern models like GPT-4o, 4.1, o1, and 5, etc.
    "gpt-oss": tiktoken.get_encoding("o200k_harmony") # Used for GPT-OSS
}

# Get the vocabulary size for each encoding
vocab_sizes = {model: encoding.n_vocab for model, encoding in encodings.items()}

# Print the vocabulary sizes
for model, size in vocab_sizes.items():
    print(f"The vocabulary size for {model} is: {size}")

The vocabulary size for gpt2 is: 50257
The vocabulary size for gpt3 is: 50281
The vocabulary size for gpt4 is: 100277
The vocabulary size for gpt4o or later is: 200019
The vocabulary size for gpt-oss is: 201088


Wow, a lot has changed throughout!

And here comes to the end of the exploration of the `tokenizer`!